# SolarScan — 07 · Démo interactive (Gradio)

Une mini-application web : on **charge une image thermique** d'un module PV → le modèle **prédit la classe** et affiche la **carte de chaleur Grad-CAM** (où il a regardé). Gradio génère un **lien public partageable**.

**Prérequis :** avoir lancé le notebook 06 et récupéré `solarscan_resnet18.pt` + `classes.json`.

> ⚡ Colab : la cellule de chargement te demandera d'**uploader** ces 2 fichiers.

## 0. Setup

In [ ]:
%pip install -q gradio torch torchvision pillow matplotlib

In [ ]:
import os, json
import numpy as np
import matplotlib
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms, models

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device :', device)

## 1. Charger le modèle sauvegardé

In [ ]:
if not os.path.exists('solarscan_resnet18.pt'):
    try:
        from google.colab import files
        print('Uploade solarscan_resnet18.pt ET classes.json :')
        files.upload()
    except Exception:
        print('Place solarscan_resnet18.pt et classes.json dans ce dossier.')

with open('classes.json') as f:
    CLASSES = json.load(f)

model = models.resnet18()
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model.load_state_dict(torch.load('solarscan_resnet18.pt', map_location=device))
model = model.to(device).eval()
print('Modele charge -', len(CLASSES), 'classes')

## 2. Prétraitement, Grad-CAM et fonction de prédiction

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
eval_tf = transforms.Compose([
    transforms.Grayscale(3), transforms.Resize((224, 224)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

def gradcam(x, target_layer):
    acts, grads = {}, {}
    h1 = target_layer.register_forward_hook(lambda m, i, o: acts.__setitem__('v', o.detach()))
    h2 = target_layer.register_full_backward_hook(lambda m, gi, go: grads.__setitem__('v', go[0].detach()))
    out = model(x)
    cls = out.argmax(1).item()
    model.zero_grad(); out[0, cls].backward()
    h1.remove(); h2.remove()
    a, g = acts['v'][0], grads['v'][0]
    cam = torch.relu((g.mean(dim=(1, 2))[:, None, None] * a).sum(0))
    return (cam / (cam.max() + 1e-8)).cpu().numpy()

def make_overlay(img, cam):
    base = np.array(img.convert('L').resize((224, 224)))
    cam_up = np.array(Image.fromarray((cam * 255).astype('uint8')).resize((224, 224))) / 255.0
    heat = (matplotlib.colormaps['jet'](cam_up)[:, :, :3] * 255).astype('uint8')
    blend = (0.5 * np.stack([base] * 3, axis=-1) + 0.5 * heat).astype('uint8')
    return Image.fromarray(blend)

def predict(img):
    if img is None:
        return {}, None
    x = eval_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(x), 1)[0].cpu().numpy()
    confidences = {CLASSES[i]: float(probs[i]) for i in range(len(CLASSES))}
    cam = gradcam(x, model.layer4[-1])
    return confidences, make_overlay(img, cam)

## 3. Lancer la démo

Un lien public (`*.gradio.live`) s'affiche : tu peux le partager. Glisse une image depuis `data/images/` pour tester.

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type='pil', label='Image thermique du module PV'),
    outputs=[gr.Label(num_top_classes=5, label='Prediction'),
             gr.Image(type='pil', label='Grad-CAM (zones regardees)')],
    title='SolarScan - Detection de defauts sur panneaux solaires',
    description='Charge une image thermique infrarouge d un module PV. Le modele predit la classe d anomalie et montre ou il a regarde.')

demo.launch(share=True)

## 4. Pour aller plus loin — déploiement permanent

Le lien Gradio est temporaire (~72 h). Pour une démo **permanente** et gratuite : déployer sur **Hugging Face Spaces** (un `app.py` avec ce code + `requirements.txt` + le fichier de poids). On pourra le faire ensemble.